# Import Libraries

In [1]:
import string
import re
import pandas as pd
from bs4 import BeautifulSoup

# Load Dataset

In [2]:
data=pd.read_csv('IMDB Dataset.csv')

# Basic EDA

In [3]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  str  
dtypes: str(2)
memory usage: 63.6 MB


In [5]:
data.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
data.shape

(50000, 2)

In [7]:
data.duplicated().sum()

np.int64(418)

In [8]:
data['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [9]:
data=data.drop_duplicates()

# Text Cleaning

In [10]:
# Lowercase

data['review']=data['review'].str.lower()

In [11]:
# Remove HTML tags

def remove_html(text):

    return BeautifulSoup(text,'html.parser').get_text()

data['review']=data['review'].apply(remove_html)

In [12]:
# Remove punctuation

def remove_punctuation(text):

    return text.translate(str.maketrans('', '',string.punctuation))

data['review']=data['review'].apply(remove_punctuation)

In [13]:
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [14]:
#find special chars

special_chars = set()

for text in data['review']:
    if isinstance(text, str):
        special_chars.update(re.findall(r'[^a-zA-Z0-9\s]', text))

print(sorted(special_chars))

['\x08', '\x10', '\x80', '\x84', '\x8d', '\x8e', '\x91', '\x95', '\x96', '\x97', '\x9a', '\x9e', '¡', '¢', '£', '¤', '¦', '§', '¨', '©', 'ª', '«', '\xad', '®', '°', '³', '´', '·', 'º', '»', '½', '¾', '¿', 'ß', 'à', 'á', 'â', 'ã', 'ä', 'å', 'æ', 'ç', 'è', 'é', 'ê', 'ë', 'ì', 'í', 'î', 'ï', 'ð', 'ñ', 'ò', 'ó', 'ô', 'õ', 'ö', 'ø', 'ù', 'ú', 'û', 'ü', 'ý', 'þ', 'č', 'ğ', 'ı', 'ō', 'ř', 'ż', 'א', 'ג', 'ו', 'י', 'כ', 'ל', 'מ', 'ן', 'ר', '‐', '–', '‘', '’', '“', '”', '…', '″', '₤', 'ℴ', 'ⅈ', '∧', '≪', '▼', '★', '、', '\uf04a', '\uf0b7', '，']


In [15]:
# Remove special characters 

def remove_special_char(text):

    return re.sub(r'[^a-zA-Z0-9\s]','',text)

data['review']=data['review'].apply(remove_special_char)

In [16]:
# Remove extra spaces

def remove_extra_spaces(text):

    return re.sub(r'\s+',' ',text).strip()

data['review']=data['review'].apply(remove_extra_spaces)

# Tokenization

In [17]:
from nltk.tokenize import word_tokenize

data['review']=data['review'].apply(word_tokenize)

In [18]:
data.head()

,review,sentiment
0,"[one, of, the, other, reviewers, has, mentione...",positive
1,"[a, wonderful, little, production, the, filmin...",positive
2,"[i, thought, this, was, a, wonderful, way, to,...",positive
3,"[basically, theres, a, family, where, a, littl...",negative
4,"[petter, matteis, love, in, the, time, of, mon...",positive


# Stopword Removal

In [19]:
from nltk.corpus import stopwords

stop_words=set(stopwords.words('english'))
data['review']=data['review'].apply(lambda words:[word for word in words if word not in stop_words])

# Lemmatization

In [20]:
import spacy


In [21]:
nlp = spacy.load("en_core_web_sm",disable=["parser", "ner"])

In [22]:
# List -> String
texts = data['review'].apply(" ".join)

# Fast Processing
docs = nlp.pipe(
    texts,
    batch_size=1000
)

# Lemmatization
data['review'] = [
    [token.lemma_ for token in doc]
    for doc in docs
]

# Feature Engineering (Vectorization)

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

data['review']=data['review'].apply(" ".join)

tfidf=TfidfVectorizer(max_features=10000,ngram_range=(1,2),sublinear_tf=True,min_df=2,max_df=0.95)

X=tfidf.fit_transform(data['review'])


In [24]:
print(type(data['review'].iloc[0]))

<class 'str'>


In [25]:
from sklearn.preprocessing import LabelEncoder

encoder=LabelEncoder()

y=encoder.fit_transform(data['sentiment'])

# Train Test Split

In [26]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

# Model Training

In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC


models={
    'logistic':LogisticRegression(random_state=42),
    'multinomialNB':MultinomialNB(),
    'svm':LinearSVC(random_state=42)
}

In [28]:
from sklearn.model_selection import cross_val_score

result=[]

for name,model in models.items():
    score=cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring='f1',
        n_jobs=-1
    ).mean()

    result.append([name,score])
    

In [46]:
df_result=pd.DataFrame(result,columns=['Name','F1_Score'])

df_result.sort_values(by='F1_Score',ascending=False)

,Name,F1_Score
0,logistic,0.893720
2,svm,0.885562
1,multinomialNB,0.866432


# Hyperparameter Tuning

In [47]:
from sklearn.model_selection import GridSearchCV

params = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs']
}

grid=GridSearchCV(
    LogisticRegression(max_iter=1000,random_state=42),
    params,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train,y_train)

print(grid.best_params_)
print(grid.best_score_)

{'C': 1, 'solver': 'lbfgs'}
0.893719809738902


# best model

In [48]:
best_model=grid.best_estimator_

In [49]:
best_model.fit(X_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [50]:
y_pred=best_model.predict(X_test)

# Evaluation

In [51]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

accuracy_score(y_test,y_pred)

0.892104467076737

In [52]:
confusion_matrix(y_test,y_pred)

array([[4362,  578],
       [ 492, 4485]])

In [53]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.90      0.88      0.89      4940
           1       0.89      0.90      0.89      4977

    accuracy                           0.89      9917
   macro avg       0.89      0.89      0.89      9917
weighted avg       0.89      0.89      0.89      9917



# Model Save

In [55]:
import joblib

joblib.dump(best_model, "models/sentiment.pkl")
joblib.dump(tfidf, "models/tfidf.pkl")

['sentiment.pkl']